# Constrained Lists (`conlist`) & Deprecations

When defining models, you sometimes need a sequence of items with a strict minimum or maximum length. While you can attempt to solve this using Union types, Pydantic provides specialized constraint functions like `conlist` to handle this cleanly.

## 1. The Union Type Problem

Imagine you want to accept a coordinate that represents either a 2D or 3D point. You might initially try using a Union of strict tuples.

```python
from pydantic import BaseModel, ValidationError

class Sphere(BaseModel):
    # Expecting either exactly 2 or exactly 3 integers
    center: tuple[int, int] | tuple[int, int, int] = (0, 0)
    radius: int = 1

try:
    # ❌ Fails: We passed 4 items
    s = Sphere(center=(1, 2, 3, 4))
except ValidationError as e:
    print(e)

```

**The Issue:** The validation error output here will be highly confusing. Because the input failed *both* conditions of the Union, Pydantic will output errors for both the 2-item tuple constraint and the 3-item tuple constraint simultaneously.

## 2. Using `conlist`

To fix the messy error reporting and simplify the logic, you can use a constrained list (`conlist`). It allows you to specify the inner type, a `min_length`, and a `max_length`.

```python
from pydantic import BaseModel, conlist

class BetterSphere(BaseModel):
    # Enforces a list of integers between length 2 and 3
    center: conlist(int, min_length=2, max_length=3) = [0, 0]
    radius: int = 1

# ✅ Valid
s1 = BetterSphere(center=[1, 2])
s2 = BetterSphere(center=(1, 2, 3)) # Tuples safely coerce to lists

# ❌ Invalid: Raises a single, clean error message:
# "List should have at most 3 items after validation, not 4"
s3 = BetterSphere(center=[1, 2, 3, 4]) 

```

## 3. Deprecation Warnings (Crucial Context)

As Pydantic evolves (specifically in V2), the way constraints are handled is shifting dramatically. You must be careful when relying on the `con*` family of functions.

### The `unique_items` Trap

Historically, `conlist` accepted a `unique_items=True` parameter to ensure no duplicate values were present. **This has been removed in modern Pydantic.**

* The official documentation suggests using the Python `set` type instead.
* **The flaw:** Sets are *unordered*. If your data requires items to be unique *and* ordered (e.g., maintaining sequence), a `set` will silently destroy your data structure. To enforce ordered uniqueness in modern Pydantic, you must write a custom `@field_validator`.

### The Shift to `Annotated`

If you read the Pydantic documentation for constraint functions like `constr` (constrained string) or `conint` (constrained integer), you will see warnings that they are heavily discouraged.

The modern best practice in Pydantic V2 is to avoid these functions entirely in favor of Python's `typing.Annotated` combined with Pydantic's `Field` object. (This will be covered in depth in a later section).



###  Interview Preparation: Constrained Lists & Deprecations

<details>
<summary><b>1. You need a field that accepts a coordinate list of either 2 or 3 integers. You define it as `center: tuple[int, int] | tuple[int, int, int]`. A user submits `[1, 2, 3, 4]`. What is the flaw with how Pydantic handles this error? What is a better approach?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
The flaw is that Pydantic will evaluate the input against both Union types independently, fail both, and return a bloated, confusing validation error stating that it failed the 2-item constraint *and* the 3-item constraint. <br><br>
A better approach is to use Pydantic's constrained list: `center: conlist(int, min_length=2, max_length=3)`. This evaluates as a single rule and returns one clean error message: "List should have at most 3 items after validation, not 4."
</details>

<details>
<summary><b>2. A legacy Pydantic codebase contains the following field: `tags: conlist(str, unique_items=True)`. Why will this break in modern Pydantic, and what is the caveat to the officially recommended fix?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
This will break because the `unique_items` parameter was deprecated and completely removed in modern Pydantic versions. <br><br>
The official documentation recommends replacing the type with a standard Python `set[str]`. However, the major caveat is that Python sets are unordered. If the original list relied on preserving the sequence of the tags, using a `set` will silently destroy that ordering. If ordered uniqueness is required, the developer must implement a custom `@field_validator` instead.
</details>

<details>
<summary><b>3. Why are specialized functions like `conint` and `constr` heavily discouraged in modern Pydantic (V2) codebases?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
These constraint functions are legacy holdovers from Pydantic V1. In Pydantic V2, the architectural standard shifted to using Python's standard `typing.Annotated` module combined with Pydantic's `Field` object or native metadata classes. Relying on `con*` functions is discouraged because they are being phased out and will eventually break in future releases.
</details>

